## The Test Document

> 📖 Read the full article: [Extracting PDF Tables on Apple Silicon: olmOCR-2 vs PaddleOCR-VL](https://codecut.ai/olmocr2-vs-paddleocr-vl/)


For a fair comparison, we will use the same PDF as part 1: the [Docling Technical Report](https://arxiv.org/pdf/2408.09869) from arXiv:

In [ ]:
import urllib.request

source = "https://arxiv.org/pdf/2408.09869"
local_pdf = "docling_report.pdf"
urllib.request.urlretrieve(source, local_pdf)

## Runtime Setup

Neither olmOCR-2 nor PaddleOCR-VL ships with native Apple Silicon support in its official Python package. Both rely on CUDA-only inference stacks. To run them on a Mac, we have two options:

1. **Rent a cloud GPU** (RunPod, Modal, Lambda) and use the official inference path
2. **Use community GGUF quantizations** with [llama.cpp](https://github.com/ggerganov/llama.cpp). GGUF is a file format that packages compressed model weights into a single file. `llama.cpp` is an inference engine that can load GGUF files and run them on Apple Silicon's GPU, bypassing the CUDA dependency entirely.

In this article, we will use the GGUF + llama.cpp path for the rest of this article because the compressed model files fit on a laptop and the setup runs free on Apple Silicon.

Install llama.cpp:

```bash
brew install llama.cpp
```

*This article uses llama.cpp build 9380.*

## olmOCR-2: Qwen2.5-VL Fine-Tune

[olmOCR-2](https://github.com/allenai/olmocr) is Allen AI's open-weight OCR model. It stands out for three reasons:

- **A 7B fine-tune of Qwen2.5-VL** reads each PDF page as an image
- **Cheap to run at scale**: on a rented NVIDIA H100, olmOCR-2 processes a few pages per second, working out to about $2 per 10,000 pages in cloud costs
- **Strongest table benchmark**: scores 84.9 on tables on its own olmOCR-Bench, the highest among open VLM-OCR models at release

olmOCR-2 takes the whole PDF page as an image and produces structured output in a single step. This is the same architecture as Docling's VLM pipeline from part 1, just with a different model.

```text
PDF page rendered as image
┌─────────────────────┐
│ Text paragraph...   │
│ Name  Score         │
│ Alice  92           │
│ Bob    85           │
└─────────────────────┘
         │
         ▼
One model reads the page
and writes the output
         │
         ▼
| Name  | Score |
|-------|-------|
| Alice | 92    |
| Bob   | 85    |
```

### Download the GGUF and vision projector

To use olmOCR-2 with llama.cpp, download two files: the model weights and the vision projector (`mmproj`).

```bash
# Language model (Q8_0, ~8 GB)
curl -L -O https://huggingface.co/lmstudio-community/olmOCR-2-7B-1025-GGUF/resolve/main/olmOCR-2-7B-1025-Q8_0.gguf

# Vision projector (F16, ~1.4 GB)
curl -L -O https://huggingface.co/lmstudio-community/olmOCR-2-7B-1025-GGUF/resolve/main/mmproj-olmOCR-2-7B-1025-F16.gguf
```

### Table extraction

olmOCR-2 reads images, not PDFs, so we'll extract tables in three steps:

1. Convert each PDF page to an image
2. Run olmOCR-2 on each image and collect the output
3. Extract the tables from the combined output with a regex

For step 1, we will use `pdf2image`, which depends on the `poppler` system binary. Install both:

```bash
brew install poppler
pip install pdf2image
```

Now convert each page to a JPEG:

In [ ]:
import subprocess
from pathlib import Path
from pdf2image import convert_from_path

images_dir = Path("images")
images_dir.mkdir(exist_ok=True)

pages = convert_from_path(local_pdf, dpi=200)
for i, page in enumerate(pages):
    page.save(images_dir / f"page_{i}.jpg")

olmOCR-2 doesn't have a pure-Python API that runs on Apple Silicon, so we shell out to `llama-mtmd-cli` via `subprocess` for each page. The command for one page looks like this:

```bash
llama-mtmd-cli \
  -m olmOCR-2-7B-1025-Q8_0.gguf \
  --mmproj mmproj-olmOCR-2-7B-1025-F16.gguf \
  --image page_0.jpg \
  -p "Convert this page to markdown. Preserve tables exactly. Output tables in HTML format." \
  --n-predict 3072
```

What each flag does:

- `-m`: the language model weights (the `.gguf` we downloaded)
- `--mmproj`: the vision encoder (the `mmproj` we downloaded)
- `--image`: the input image to process
- `-p`: the prompt sent to the model
- `--n-predict`: the maximum number of tokens to generate (3072 is enough for most table-heavy pages)

Wrap it in a Python helper so we can loop over pages: